# Hydrological Ensemble Verification with _veriflow_

Forecast verification is essential for evaluating how well model predictions match observations and how forecast skill changes with lead time. It provides insight into both the accuracy and reliability of forecasts, helping users understand model performance and make informed decisions.

**veriflow** is a Python-based framework for evaluating forecast performance using a configuration-driven workflow. With _veriflow_, users can process data, compute verification metrics, and analyse results in a consistent and reproducible way.

This notebook demonstrates how to perform hydrometeorological forecast verification using _veriflow_. It guides you through setting up a verification pipeline, running it on NetCDF input data, and exploring and interpreting the results interactively.

In this example, we compare multiple discharge forecast variants against observations and analyse their performance using interactive visualizations.

By the end of this notebook, you will be able to:
- Understand what _veriflow_ is and how it can be used for forecast verification  
- Configure a verification pipeline using _veriflow_  
- Run the pipeline on local NetCDF datasets  
- Inspect and interpret the resulting verification outputs  
- Explore forecast performance using interactive visualizations  
- Understand the value of hydrometeorological forecast verification  

---


## Dataset

This notebook uses hydrometeorological data for the River Rhine basin. The datasets consist of both observations and ensemble-based forecasts.

### Data description

The original dataset includes:

- **Ensemble meteorological forecasts**  
  5-member ECMWF reforecast ensembles of precipitation and temperature

- **Hydrological model outputs**  
  Discharge ensemble forecasts generated by forcing a hydrological model (HBV) with the meteorological ensembles

- **Observations**  
  Corresponding discharge observations used for verification

The datasets cover:
- **134 HBV catchments** in the Rhine basin  
- **~88 discharge locations** used in operational forecasting systems (e.g. Delft-FEWS)

### Variables and temporal coverage

In this notebook, we focus on **discharge (Q)**, which is commonly used in hydrological forecast verification.

The observation dataset has a **daily temporal resolution** and covers the period from 1961-01-03 to 2008-01-01  

Using a long historical period allows us to evaluate forecast performance across many events, stations, and forecast lead times.

The same workflow can be applied to other variables (e.g. precipitation and temperature), depending on the verification objective and available observation data.

### Data download

The datasets used in this notebook are available for download via  
<span style="color: red;">[DOWNLOAD LINK to be added HERE]</span>.

After downloading, place the files in a local `data/` directory within the repository.

### Data source

The dataset was originally used in:

> Verkade et al. (2013), *Post-processing ECMWF precipitation and temperature ensemble reforecasts for operational hydrologic forecasting at various spatial scales*, Journal of Hydrology.  
> https://doi.org/10.1016/j.jhydrol.2013.07.039  

Additional information on data are available via the 4TU.ResearchData repository:  
https://data.4tu.nl/articles/_/12694658/1

### Scientific context

These data were originally prepared for use in the *Ensemble Verification System* developed by the US National Weather Service.

In the study of Verkade et al. (2013):
- Biases in meteorological ensemble forecasts (precipitation and temperature) are analysed  
- These biases are propagated through a hydrological model to generate discharge forecasts  
- Multiple post-processing techniques are applied to improve forecast skill  

These include:
- Quantile-to-quantile transformation  
- Linear regression approaches  
- Logistic regression  

The resulting discharge ensembles are evaluated using multiple verification metrics (e.g. CRPS, Brier score, ROC), providing insight into both forecast accuracy and reliability.

-----


## Experiment setup

In this notebook, we evaluate how different post-processing methods affect forecast performance by comparing multiple forecast variants against observations.

### Forecast variants

We compare three forecast variants against **observations (`obs`)**:

- **raw_raw**  
  Baseline forecast without post-processing. This represents the original model output driven by raw meteorological ensembles.

- **lin_log**  
  Forecast with a linear-log transformation. This method aims to correct bias by modelling relationships under an assumed statistical structure, and is often used for skewed hydrological variables, such as discharge.

- **qqt_qqt**  
  Forecast with a quantile-to-quantile transformation. This method adjusts the distribution of forecasts to better match observed statistics.

Each forecast variant represents a different post-processing strategy, allowing us to evaluate how these methods affect forecast performance.

### Verification setup

The verification is performed over a defined time period and evaluated across multiple forecast **lead times** (forecast horizons).

The configuration includes:
- Comparison of forecasts against observations  
- Evaluation across multiple lead times 
- Computation of standard verification metrics (e.g. accuracy and reliability)

This setup allows us to assess:
- How forecast skill chnages with increasing lead time  
- How bias correction and post-processing influence forecast quality  

---

## Imports and environment setup

We start by importing the required Python libraries and _veriflow_ components used throughout this notebook.

These include:
- General Python utilities for handling dates and file paths  
- _veriflow_ configuration objects used to define the verification setup  
- Data source definitions for loading NetCDF input data  
- Scoring methods used to evaluate forecast performance  
- The pipeline function that executes the verification workflow  
- The create_app function that launches an interactive dashboard for exploring the results.

It's also possible to activate automatic reloading of modules to make development and iteration easier.

In [ ]:
# Add automatic reloading of modules in case of changes
%load_ext autoreload
%autoreload 2

from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Local Dash app for interactive visualization
from app import create_app

from veriflow.configuration import Config, GeneralInfoConfig
from veriflow.configuration.utils import (
    LeadTimes,
    Range,
    TimeUnits,
    VerificationPair,
    VerificationPeriod,
)
from veriflow.constants import DataType
from veriflow.datasources import ZarrConfig
from veriflow.pipeline import run_pipeline
from veriflow.scores import CrpsForEnsembleConfig, RankHistogramConfig

In [ ]:
general = GeneralInfoConfig(
    # Evaluate forecasts issued during this period.
    # The period is defined over the forecast_reference_time dimension.
    verification_period=VerificationPeriod(
        start=datetime(1998, 1, 1, tzinfo=timezone.utc),
        end=datetime(2008, 12, 31, tzinfo=timezone.utc),
        dimension="forecast_reference_time",
    ),
    # Define which forecast datasets are compared against observations.
    verification_pairs=[
        VerificationPair(id="raw_raw", obs="obs", sim="raw_raw", variable="Q"),
        VerificationPair(id="lin_log", obs="obs", sim="lin_log", variable="Q"),
        VerificationPair(id="qqt_qqt", obs="obs", sim="qqt_qqt", variable="Q"),
    ],
    # Evaluate lead times from 0 to 10 days.
    lead_times=LeadTimes(
        unit=TimeUnits.day,
        values=Range(start=0, end=10, step=1),
    ),
)

In [ ]:
config = Config(
    fileversion="0.1.0",
    general=general,
    datasources=[
        ZarrConfig(
            general=general,
            import_adapter="zarr",
            source="obs",
            data_type=DataType.observed_historical,
            consolidated=True,
            path="https://s3.deltares.nl/deltares-verification-assets/rhine_dataset_verkade_2013/obs_Q.zarr",
        ),
        ZarrConfig(
            general=general,
            import_adapter="zarr",
            source="lin_log",
            data_type=DataType.simulated_forecast_ensemble,
            consolidated=True,
            path="https://s3.deltares.nl/deltares-verification-assets/rhine_dataset_verkade_2013/case-lin-log_Q.zarr",
        ),
        ZarrConfig(
            general=general,
            import_adapter="zarr",
            source="raw_raw",
            data_type=DataType.simulated_forecast_ensemble,
            consolidated=True,
            path="https://s3.deltares.nl/deltares-verification-assets/rhine_dataset_verkade_2013/case-raw-raw_Q.zarr",
        ),
        ZarrConfig(
            general=general,
            import_adapter="zarr",
            source="qqt_qqt",
            data_type=DataType.simulated_forecast_ensemble,
            consolidated=True,
            path="https://s3.deltares.nl/deltares-verification-assets/rhine_dataset_verkade_2013/case-qqt-qqt_Q.zarr",
        ),
    ],
    scores=[
        # CRPS: keep all dimensions for detailed analysis (e.g. per lead time, station)
        CrpsForEnsembleConfig(
            score_adapter="crps_for_ensemble",
            general=general,
            reduce_dims=[],
        ),
        # Rank histogram: aggregate over time to assess overall ensemble calibration
        RankHistogramConfig(
            score_adapter="rank_histogram",
            general=general,
            reduce_dims=["forecast_reference_time"],
        ),
    ],
)

In [ ]:
# The entire workflow can be executed with a single function call:
import logging

# Set up logging to display INFO and above messages in the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[logging.StreamHandler()]
)

output_dataset = run_pipeline(config)

In [ ]:
output_dataset.get(output_dataset.verification_pairs[0])

In [ ]:
from app import create_app
app = create_app(output_dataset)
app.run(jupyter_mode="inline", jupyter_height=1400)

## Defining the verification configuration

_veriflow_ uses a configuration-driven workflow to define and run the verification pipeline. 
<span style="color: red;">[Should there be a connection to the doc/main page?]</span>.

In this notebook, the configuration is divided into three main components:

- **Data sources**: define where the input data comes from  
- **General settings**: define the verification period, forecast lead times, and forecast–observation pairs  
- **Scores**: define which verification metrics are computed  

We define each of these components step by step below.

---

## Data sources (input data)

Data sources define how _veriflow_ loads the input data used in the verification.

In this example, we use:
- One observation dataset (`obs`)  
- Three forecast datasets (`raw_raw`, `lin_log`, `qqt_qqt`)  

Each dataset is stored as a NetCDF file and loaded from a local `data/` directory.

These datasets are expected to follow the _veriflow_ data model, meaning they contain standardized dimensions and coordinates (e.g. time, station, variable). This allows the pipeline to automatically align observations and forecasts during verification.

### Import the input data

### Inspecting the input data

Before configuring the verification pipeline, we first inspect the input datasets.

Try to answer the following questions while looking at the summary below:

1. What variable is verified in this example?
2. What is the temporal coverage of the observation and forecast data?
3. How many stations are available for the observation and forecast data?
4. How do the observation and forecast datasets differ?
5. How many forecast lead times and ensemble members are available?

This quick inspection provides context for the rest of the notebook and helps you understand the data before running the verification workflow.

In [ ]:
# Files used in this notebook
file_map = {
    "obs": data_dir / "obs_Q.nc",
    "raw_raw": data_dir / "case-raw-raw_Q.nc",
    "lin_log": data_dir / "case-lin-log_Q.nc",
    "qqt_qqt": data_dir / "case-qqt-qqt_Q.nc",
}

# Open datasets
datasets = {name: xr.open_dataset(path) for name, path in file_map.items()}

obs_ds = datasets["obs"]
raw_ds = datasets["raw_raw"]
linlog_ds = datasets["lin_log"]
qqt_ds = datasets["qqt_qqt"]


def summarize_dataset(name, ds):
    variables = list(ds.coords["variable"].values) if "variable" in ds.coords else []
    stations = ds.sizes.get("station")
    realizations = ds.sizes.get("realization")

    if "time" in ds.coords:
        time_coord = ds["time"]
        if time_coord.ndim == 1:
            start = str(time_coord.min().values)[:10]
            end = str(time_coord.max().values)[:10]
            n_time = ds.sizes.get("time")
        else:
            start = str(time_coord.min().values)[:10]
            end = str(time_coord.max().values)[:10]
            n_time = "derived"
    else:
        start = end = n_time = None

    lead_times_days = None
    if "lead_time" in ds.coords:
        lead_times_days = (ds.forecast_period.values / np.timedelta64(1, "D")).astype(int).tolist()

    return {
        "dataset": name,
        "file": file_map[name].name,
        "variables": variables,
        "stations": stations,
        "time_steps": n_time,
        "start": start,
        "end": end,
        "dimensions": tuple(ds.dims),
        "ensemble_members": realizations,
        "lead_times_days": lead_times_days,
    }


summary_df = pd.DataFrame(summarize_dataset(name, ds) for name, ds in datasets.items())

summary_df

#### What do we learn from this inspection?

The summary shows that:

- The verified variable is **discharge (`Q`)**.
- The observation dataset is a daily time series with dimensions such as `variable`, `time`, and `station`.
- The forecast datasets include additional dimensions:
  - `forecast_reference_time`
  - `forecast_period`
  - `realization`
- These additional dimensions represent the ensemble forecast structure:
  - Forecasts are issued at different reference times
  - Each forecast has multiple lead times
  - Each forecast contains multiple ensemble members
- In this example, the forecast lead times range from **0 to 10 days**.
- Each forecast contains **5 ensemble members**.

This structure is typical for ensemble forecast verification and is one reason why a standardized pipeline such as _veriflow_ is useful.

---

### Example-location preview

Before running the verification pipeline, we inspect the data at one example station.
<span style="color: red;">[Which station should we use?]</span>.

This preview focuses on:
- The seasonal behaviour of observed discharge
- Forecast behaviour during a flood period
- Forecast behaviour during a drought period
- Differences between lead times and forecast variants

#### Choose station and prepare data

In [ ]:
# Choose one example station
example_station = str(obs_ds.station.values[0])

# Observation series
obs_series = obs_ds["data"].sel(variable="Q", station=example_station)

# Forecast datasets
forecast_datasets = {
    "raw_raw": raw_ds["data"].sel(variable="Q", station=example_station),
    "lin_log": linlog_ds["data"].sel(variable="Q", station=example_station),
    "qqt_qqt": qqt_ds["data"].sel(variable="Q", station=example_station),
}

example_station

#### Observation plot: all years by day-of-year + long-term average timeseries 

In [ ]:
# Convert observation series to DataFrame
obs_df = obs_series.to_dataframe(name="Q").reset_index()
obs_df["year"] = obs_df["time"].dt.year
obs_df["dayofyear"] = obs_df["time"].dt.dayofyear

# Mean seasonal cycle
mean_daily_q = obs_df.groupby("dayofyear")["Q"].mean()

# Create figure with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

# --- Top plot: full time series ---
obs_series.plot(
    ax=axes[0],
    color="black",
    linewidth=1,
)

axes[0].set_title(f"Observed discharge (full period) at station {example_station}")
axes[0].set_ylabel("Discharge")
axes[0].grid(True, alpha=0.3)

# --- Bottom plot: seasonal cycle ---
for year, group in obs_df.groupby("year"):
    axes[1].plot(
        group["dayofyear"],
        group["Q"],
        alpha=0.15,
        linewidth=0.8,
    )

# Mean line
axes[1].plot(
    mean_daily_q.index,
    mean_daily_q.values,
    color="black",
    linewidth=2.5,
    label="Mean daily discharge",
)

axes[1].set_title("Seasonal cycle (all years)")
axes[1].set_xlabel("Day of year")
axes[1].set_ylabel("Discharge")
axes[1].set_xlim(1, 366)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

#### Helper function for event plots

In [ ]:
def forecast_stats_for_lead(ds, lead_days):
    """Return ensemble mean, min and max for a selected lead time."""
    selected = ds.sel(forecast_period=np.timedelta64(lead_days, "D"))

    mean = selected.mean(dim="realization")
    lower = selected.min(dim="realization")
    upper = selected.max(dim="realization")

    # Use valid time as plotting dimension
    mean = mean.swap_dims({"forecast_reference_time": "time"})
    lower = lower.swap_dims({"forecast_reference_time": "time"})
    upper = upper.swap_dims({"forecast_reference_time": "time"})

    return mean, lower, upper


def plot_event_forecasts(
    obs_series,
    forecast_datasets,
    period,
    period_label,
    lead_times=(1, 5, 10),
):
    """Plot observed discharge and forecast ensemble summaries for multiple lead times."""
    colors = {
        "raw_raw": "tab:blue",
        "lin_log": "tab:orange",
        "qqt_qqt": "tab:green",
    }

    fig, axes = plt.subplots(
        len(lead_times),
        1,
        figsize=(14, 10),
        sharex=True,
        sharey=False,
    )

    obs_period = obs_series.sel(time=period)

    for ax, lead_days in zip(axes, lead_times):
        # Observation
        obs_period.plot(
            ax=ax,
            color="black",
            linewidth=2,
            label="Observed",
        )

        # Forecast variants
        for name, ds in forecast_datasets.items():
            mean, lower, upper = forecast_stats_for_lead(ds, lead_days)

            mean_period = mean.sel(time=period)
            lower_period = lower.sel(time=period)
            upper_period = upper.sel(time=period)

            color = colors[name]

            ax.fill_between(
                mean_period["time"].values,
                lower_period.values,
                upper_period.values,
                color=color,
                alpha=0.18,
            )

            mean_period.plot(
                ax=ax,
                color=color,
                linewidth=1.8,
                label=f"{name} mean",
            )

        ax.set_title(f"{period_label} | lead time = {lead_days} days")
        ax.set_ylabel("Discharge")
        ax.grid(True, alpha=0.3)

    axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("Date")

    plt.suptitle(
        f"Observed and forecast discharge at station {example_station}",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

#### Flood-period plot (1995)

This period corresponds to a major high-water event in the Rhine basin, driven by heavy rainfall and snowmelt upstream.

Water levels reached near-record values, leading to the evacuation of around 250,000 people in the Netherlands due to the risk of dike failure <span style="color: red;">[proper citation needed]</span>. 

Such events are particularly relevant for forecast verification, as they test how well models capture extreme discharge conditions and peak timing.


In [ ]:
flood_period = slice("1995-01-01", "1995-04-01")

plot_event_forecasts(
    obs_series=obs_series,
    forecast_datasets=forecast_datasets,
    period=flood_period,
    period_label="Flood period: 1 Jan 1995 to 1 Apr 1995",
    lead_times=(1, 5, 10),
)

#### Drought-period plot (2003)

This period corresponds to a major drought event in the Rhine basin, which is associated with the 2003 European heatwave and prolonged dry conditions.

Discharge levels were significantly lower than average, reflecting low precipitation and high evaporation <span style="color: red;">[proper citation needed]</span>. 

Such events are relevant for forecast verification, as they test how well models capture low-flow conditions, including the magnitude and persistence of drought.

In [ ]:
drought_period = slice("2003-07-01", "2003-10-01")

plot_event_forecasts(
    obs_series=obs_series,
    forecast_datasets=forecast_datasets,
    period=drought_period,
    period_label="Drought period: 1 Jul 2003 to 1 Oct 2003",
    lead_times=(1, 5, 10),
)

#### Questions to consider

<span style="color: red;">[The answers below are intentionally generic, so that they are still correct if users changes a station. Should it be more station-specefic?]</span>

Use the plots above to inspect the input data before running the verification pipeline.

1. **How does observed discharge vary throughout the year?**  
   The observation plot shows strong seasonal variability. Some periods have consistently higher discharge, while other periods show lower-flow conditions.

2. **How does forecast behaviour change with lead time?**  
   Forecasts generally become less precise as lead time increases. This can be seen as larger deviations from observations and changes in the ensemble spread.

3. **What does the shaded forecast range represent?**  
   The shaded area shows the minimum-to-maximum range of the ensemble members. A wider shaded area indicates larger ensemble spread and therefore higher forecast uncertainty.

4. **Do the post-processed forecasts behave differently from the raw forecast?**  
   Yes. The `lin_log` and `qqt_qqt` forecasts can differ from `raw_raw` in both mean behaviour and ensemble spread. These differences indicate how post-processing changes the forecast distribution.

5. **Why compare both flood and drought periods?**  
   Forecast performance can vary under different hydrological conditions. A method that performs well during high-flow events may not perform equally well during low-flow periods.

6. **Why is this useful before verification?**  
   Visual inspection helps you understand the structure and behaviour of the input data. The formal verification scores computed later provide a more systematic comparison across stations, lead times, and forecast variants.

---


## General verification settings

After inspecting the data, we now define how the verification will be performed.

In _veriflow_, this is done through a **general configuration**, which controls the structure of the verification process.

This includes:
- the **verification period**: the time window over which forecasts are evaluated  
- the **forecast periods**: the lead times (e.g. 1–10 days ahead)  
- the **verification pairs**: which forecast datasets are compared against observations  

Each **verification pair** represents a comparison between one forecast variant (e.g. `raw_raw`, `lin_log`, `qqt_qqt`) and the observations.

Together, these settings define *what is being compared*, *over which time period*, and *at which forecast horizons*.

In this configuration, we evaluate forecasts issued between **1 March 1990** and **31 July 2010**.

The verification is performed for forecast lead times from **0 to 10 days**, and three forecast variants are compared against the same observation dataset.

---

## Verification scores

Verification scores quantify how well the forecasts match the observations.

In this example, we compute two commonly used metrics for ensemble forecasts:

- **Continuous Ranked Probability Score (CRPS)**  
  A measure of probabilistic forecast accuracy. It evaluates how well the predicted distribution matches the observed value.  
  Lower values indicate better performance.

- **Rank Histogram**  
  A diagnostic tool used to assess ensemble calibration and spread.  
  It shows how often the observation falls within the ensemble distribution.

These scores are computed for each forecast variant, across all stations and forecast lead times.

<span style="color: red;">[What other score would be nice to compute for this example data set?]</span>

---

## Running the verification pipeline

With the configuration in place, we can now run the verification pipeline.

This is where _veriflow_ brings everything together: the input data, the verification setup, and the selected scores.

Running the pipeline will:
1. Load and align the input datasets  
2. Match forecasts and observations across stations, time, and lead times  
3. Compute the requested verification scores  
4. Return the results in a standardized output object  

The result is stored in an **`OutputDataset`**, which contains all computed scores and can be further analysed or visualized.

---

## Inspecting and exploring the output

The pipeline returns an **`OutputDataset`**, which provides a standardized interface to all verification results.

This object contains:
- the defined **verification pairs**
- the computed **scores**
- the aligned datasets used during verification  

Each **verification pair** represents a comparison between one forecast dataset and the observations.

To inspect the results, we first select a verification pair and then extract the corresponding `xarray.Dataset`. This allows us to explore the data programmatically and use it for further analysis or visualization.

The extracted dataset provides access to:
- forecast values  
- observations  
- computed scores  
- coordinates such as station, variable, and forecast period  

### Check the type of object returned by the pipeline

In [ ]:
type(output_dataset)

### Show the available verification pairs

In [ ]:
output_dataset.verification_pairs

### Select one verification pair

The `OutputDataset` stores results per verification pair.  
To inspect the results, we select one pair and extract the corresponding dataset.

In [ ]:
# Select the pair with id "raw_raw"
pair_id = "raw_raw"
verification_pair = [p for p in output_dataset.verification_pairs if p.id == pair_id][0]

ds = output_dataset.get(verification_pair=verification_pair)
ds

### Inspect variables and coordinates for one verification pair

In [ ]:
list(ds.data_vars)

In [ ]:
list(ds.coords)

### Extracting verification scores

The extracted dataset contains multiple variables, including forecasts, observations, and computed scores.

Here, we briefly extract the **CRPS** and **rank histogram** results to prepare for visualization.

#### Extracting CRPS

The CRPS results are stored as a multi-dimensional array, including station, time, and lead time.

To get a clearer overview, we aggregate the scores across stations and time, leaving only the dependence on forecast lead time.

In [ ]:
crps = ds["crps_for_ensemble"]

# Aggregate over space and time
crps_mean = crps.mean(dim=["station", "forecast_reference_time"])

crps_mean

As you can see, the CRPS values increase with forecast lead time, indicating that forecast accuracy decreases as we predict further into the future.

This is expected behaviour: uncertainty grows with lead time, making forecasts less precise. The relatively low CRPS at short lead times suggests good short-term performance, while higher values at longer lead times reflect increasing forecast error.

---

## Interactive visualization

After inspecting the output programmatically, we can explore the verification results interactively using a Dash application.

The app makes it possible to:
- Compare forecast variants side by side
- Select stations and variables
- Inspect results across forecast lead times
- Explore both forecast behaviour and verification scores

Two main views are currently available:

### Scatter plot

The scatter plot compares observed and forecast values.

- Points close to the **1:1 line** indicate good agreement
- Systematic deviations from the line may indicate bias
- The spread of points gives insight into forecast uncertainty

### CRPS plot

The CRPS plot shows forecast performance as a function of lead time.

- Lower CRPS values indicate better performance
- Increasing CRPS with lead time is expected
- Differences between forecast variants show how post-processing affects forecast skill

<span style="color: red;">[Note: if the app still has a rank-histogram tab later, this markdown should also mention it.]</span>

### Interpretation of results

<span style="color: red;">[Note: Currently, interpretations are generic. We can add more specific explanation after picking the station to focus on.]</span>

### Scatter plot
- Points close to the diagonal line indicate good agreement
- Spread of points reflects forecast uncertainty
- Bias can be observed as systematic deviation from the line

### CRPS
- Lower CRPS values indicate better forecast skill
- Increasing CRPS with lead time is expected
- Differences between methods show which approach performs better

### Rank histogram
- A flat histogram indicates a well-calibrated ensemble
- U-shaped distributions suggest under-dispersion
- Dome-shaped distributions suggest over-dispersion

These diagnostics provide insight into both accuracy and reliability of the forecasts.

---


## Optional: Using YAML-based configuration instead of Python 

In this notebook, we used Python objects to define the configuration.

_veriflow_ also supports YAML-based configuration files, which:
- improve reproducibility
- make pipelines easier to share
- simplify deployment in production systems

Both approaches produce identical results.

---
